# Clinical Genomics: From Variants to Patient Care

## Tier 3 - Applied Bioinformatics

This notebook covers the principles and practice of clinical genomics: how genetic variants are classified, interpreted, and used to guide patient care. Inspired by the Medical Genomics course (V.E. Ramensky, A.A. Zharikova, MSU).

### Learning Objectives
- Understand the scope of clinical genomics and types of genetic testing
- Apply ACMG/AMP variant classification guidelines
- Query clinical databases (ClinVar, gnomAD) programmatically
- Evaluate in silico variant effect predictors (SIFT, PolyPhen-2, CADD, REVEL, AlphaMissense)
- Understand pharmacogenomics, cancer genomics, and polygenic risk scores
- Recognize ethical considerations in clinical genetic testing

### Prerequisites
- Module 01: NGS Fundamentals
- Module 02: Variant Calling and SNP Analysis
- Basic understanding of Mendelian genetics

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Deep Learning for Biology](../../Tier_3_Applied_Bioinformatics/10_Deep_Learning_for_Biology/01_deep_learning_for_biology.ipynb) | [Next: Tier 4 Skills Check: Algorithms & Data Structures →](../../Tier_4_Algorithms_and_Data_Structures/00_Skills_Check/00_skills_check.ipynb)

---

## 1. Introduction to Clinical Genomics

### 1.1 Precision Medicine and Genomics

**Precision medicine** (sometimes called personalized medicine) uses an individual's genomic information to guide clinical decisions -- from diagnosis to treatment selection to risk assessment.

The journey from the Human Genome Project (completed 2003) to routine clinical sequencing has been driven by:
- **Dramatic cost reduction**: Whole-genome sequencing dropped from ~$100M (2001) to ~$200 (2024)
- **Improved interpretation**: Growing databases of variant-disease associations
- **Clinical validation**: Proven utility in rare disease diagnosis, cancer treatment, pharmacogenomics

```
Precision Medicine Pipeline:

  Patient         Sequencing        Bioinformatics       Clinical
  Sample    --->   (WGS/WES/    --->  Variant       --->  Interpretation
  (blood,          Panel)            Calling &            & Report
   tumor)                            Annotation
                                         |
                                         v
                                    Databases:
                                    ClinVar, OMIM,
                                    gnomAD, PharmGKB
                                         |
                                         v
                                    Treatment
                                    Decision
```

### 1.2 Types of Genetic Testing

| Type | Purpose | Examples | Typical Approach |
|------|---------|---------|------------------|
| **Diagnostic** | Identify cause of existing disease | Rare disease workup, syndromic child | WES/WGS or gene panels |
| **Predictive/Presymptomatic** | Assess future disease risk | BRCA1/2 for breast cancer, Huntington's | Targeted testing |
| **Carrier testing** | Identify heterozygous carriers of recessive conditions | Cystic fibrosis, sickle cell | Carrier panels |
| **Pharmacogenomic** | Guide drug selection/dosing | Warfarin dosing, 5-FU toxicity risk | PGx panels |
| **Prenatal/Newborn** | Screen or diagnose in fetus/newborn | NIPT, newborn screening panels | cfDNA, targeted panels |
| **Somatic/Tumor** | Guide cancer treatment | Actionable mutations in tumors | Tumor panels, WES |

### 1.3 Clinical vs Research Sequencing

Clinical and research sequencing differ in critical ways:

| Aspect | Clinical (CLIA/CAP) | Research |
|--------|-------------------|----------|
| Regulation | CLIA-certified lab required | No specific lab certification |
| Validation | Analytically validated assay | Method may be exploratory |
| Reporting | Standardized clinical report | Varies widely |
| Turnaround | Defined TAT (often 2-4 weeks) | Variable |
| Variants reported | Clinically actionable only | All variants of interest |
| Confirmatory testing | Orthogonal confirmation (Sanger) | Not required |
| Return of results | Duty to report to patient | Not always returned |

In [ ]:
# Imports for the entire notebook
import json
import math
from collections import defaultdict
from typing import NamedTuple

# Optional: for ClinVar API queries (Section 3)
try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False
    print("requests not installed -- ClinVar API examples will use mock data")

print("Imports ready.")

---

## 2. Variant Classification (ACMG/AMP Guidelines)

The **American College of Medical Genetics and Genomics (ACMG)** and the **Association for Molecular Pathology (AMP)** published landmark guidelines in 2015 (Richards et al., *Genetics in Medicine*) for classifying sequence variants. These guidelines are the global standard for clinical variant interpretation.

### 2.1 The 5-Tier Classification System

Every variant evaluated in a clinical context is assigned to one of five categories:

```
ACMG 5-Tier Classification:

  Pathogenic  >  Likely       >  Variant of    >  Likely   >  Benign
     (P)         Pathogenic      Uncertain         Benign      (B)
                   (LP)          Significance       (LB)
                                   (VUS)

  |-- Reportable as   --|  |-- Typically not  --|  |-- Reportable as  --|
     disease-causing        acted upon clinically     not disease-causing
```

- **Pathogenic (P)**: Strong evidence that the variant causes disease
- **Likely Pathogenic (LP)**: >90% certainty of being disease-causing
- **VUS**: Insufficient evidence to classify -- the most challenging category
- **Likely Benign (LB)**: >90% certainty of being benign
- **Benign (B)**: Strong evidence that the variant does not cause disease

### 2.2 ACMG/AMP Evidence Categories

The classification is based on accumulating evidence from multiple categories:

**1. Population data**
- Allele frequency in large reference populations (gnomAD, ExAC)
- If a variant is common (e.g., MAF > 5%), it is almost certainly benign (BA1 criterion)
- Absence from population databases supports pathogenicity (PM2)

**2. Computational/predictive data**
- In silico predictions (SIFT, PolyPhen-2, CADD, REVEL)
- Conservation across species
- Protein domain/structural impact

**3. Functional data**
- Well-established functional assays showing a damaging effect
- Functional studies in model organisms

**4. Segregation data**
- Co-segregation with disease in affected families
- De novo occurrence in affected individual with confirmed parentage

**5. De novo / allelic data**
- De novo variants in a patient with disease and no family history
- Variant detected in trans with a known pathogenic variant (compound heterozygosity)

**6. Other data**
- Variant reported as pathogenic by reputable source
- Patient phenotype highly specific for gene

### 2.3 ACMG/AMP Criteria Codes

Each piece of evidence is assigned a **code** with a **strength level**:

**Pathogenic criteria:**

| Strength | Codes | Examples |
|----------|-------|----------|
| Very Strong | PVS1 | Null variant in a gene where loss-of-function is a known disease mechanism |
| Strong | PS1-PS4 | Same amino acid change as established pathogenic variant (PS1); de novo with confirmed parentage (PS2); well-established functional study (PS3); prevalence in affecteds vs controls (PS4) |
| Moderate | PM1-PM6 | Located in mutational hotspot (PM1); absent from population databases (PM2); protein length change (PM4); novel missense in gene with low missense rate (PM5); assumed de novo (PM6) |
| Supporting | PP1-PP5 | Co-segregation (PP1); computational evidence (PP3); patient phenotype specific for gene (PP4); reputable source reports pathogenic (PP5) |

**Benign criteria:**

| Strength | Codes | Examples |
|----------|-------|----------|
| Stand-alone | BA1 | Allele frequency >5% in any population |
| Strong | BS1-BS4 | Frequency greater than expected for disorder (BS1); observed in healthy adult (BS2); functional study shows no effect (BS3); non-segregation with disease (BS4) |
| Supporting | BP1-BP7 | Missense in gene where only truncating cause disease (BP1); in trans with pathogenic variant in dominant gene (BP2); in silico predictions suggest benign (BP4); synonymous with no splice impact (BP7) |

In [ ]:
# ACMG classification rules (simplified)
# Based on Richards et al. 2015, Table 5

CRITERIA_STRENGTH = {
    # Pathogenic criteria
    'PVS1': 'very_strong',
    'PS1': 'strong', 'PS2': 'strong', 'PS3': 'strong', 'PS4': 'strong',
    'PM1': 'moderate', 'PM2': 'moderate', 'PM3': 'moderate',
    'PM4': 'moderate', 'PM5': 'moderate', 'PM6': 'moderate',
    'PP1': 'supporting', 'PP2': 'supporting', 'PP3': 'supporting',
    'PP4': 'supporting', 'PP5': 'supporting',
    # Benign criteria
    'BA1': 'stand_alone',
    'BS1': 'strong', 'BS2': 'strong', 'BS3': 'strong', 'BS4': 'strong',
    'BP1': 'supporting', 'BP2': 'supporting', 'BP3': 'supporting',
    'BP4': 'supporting', 'BP5': 'supporting', 'BP6': 'supporting',
    'BP7': 'supporting',
}


def classify_variant(criteria: list[str]) -> str:
    """Classify a variant based on ACMG/AMP criteria codes.
    
    Implements the combining rules from Richards et al. 2015, Table 5.
    Returns one of: Pathogenic, Likely Pathogenic, VUS, Likely Benign, Benign.
    """
    path_criteria = [c for c in criteria if c.startswith(('PVS', 'PS', 'PM', 'PP'))]
    benign_criteria = [c for c in criteria if c.startswith(('BA', 'BS', 'BP'))]
    
    # Count by strength
    pvs = sum(1 for c in path_criteria if CRITERIA_STRENGTH.get(c) == 'very_strong')
    ps = sum(1 for c in path_criteria if CRITERIA_STRENGTH.get(c) == 'strong')
    pm = sum(1 for c in path_criteria if CRITERIA_STRENGTH.get(c) == 'moderate')
    pp = sum(1 for c in path_criteria if CRITERIA_STRENGTH.get(c) == 'supporting')
    
    ba = sum(1 for c in benign_criteria if CRITERIA_STRENGTH.get(c) == 'stand_alone')
    bs = sum(1 for c in benign_criteria if CRITERIA_STRENGTH.get(c) == 'strong')
    bp = sum(1 for c in benign_criteria if CRITERIA_STRENGTH.get(c) == 'supporting')
    
    # Benign rules (checked first -- BA1 alone is stand-alone benign)
    if ba >= 1:
        return 'Benign'
    if bs >= 2:
        return 'Benign'
    if bs >= 1 and bp >= 1:
        return 'Likely Benign'
    if bp >= 2:
        return 'Likely Benign'
    
    # Pathogenic rules
    # Pathogenic: (i) 1 Very Strong AND (>=1 Strong OR >=2 Moderate OR 1 Moderate+1 Supporting OR >=2 Supporting)
    if pvs >= 1:
        if (ps >= 1 or pm >= 2 or (pm >= 1 and pp >= 1) or pp >= 2):
            return 'Pathogenic'
    # Pathogenic: (ii) >=2 Strong
    if ps >= 2:
        return 'Pathogenic'
    # Pathogenic: (iii) 1 Strong AND (>=3 Moderate OR 2 Moderate+>=2 Supporting OR 1 Moderate+>=4 Supporting)
    if ps >= 1:
        if (pm >= 3 or (pm >= 2 and pp >= 2) or (pm >= 1 and pp >= 4)):
            return 'Pathogenic'
    
    # Likely Pathogenic
    # (i) 1 Very Strong AND 1 Moderate
    if pvs >= 1 and pm >= 1:
        return 'Likely Pathogenic'
    # (ii) 1 Strong AND 1-2 Moderate
    if ps >= 1 and 1 <= pm <= 2:
        return 'Likely Pathogenic'
    # (iii) 1 Strong AND >=2 Supporting
    if ps >= 1 and pp >= 2:
        return 'Likely Pathogenic'
    # (iv) >=3 Moderate
    if pm >= 3:
        return 'Likely Pathogenic'
    # (v) 2 Moderate AND >=2 Supporting
    if pm >= 2 and pp >= 2:
        return 'Likely Pathogenic'
    # (vi) 1 Moderate AND >=4 Supporting
    if pm >= 1 and pp >= 4:
        return 'Likely Pathogenic'
    # (vii) 1 Very Strong alone -> LP
    if pvs >= 1:
        return 'Likely Pathogenic'
    
    return 'VUS'


# Test with examples
test_cases = [
    (['PVS1', 'PM2'],                'Null variant + absent from populations'),
    (['PS1', 'PM2', 'PP3'],          'Same AA change + absent + computational'),
    (['PS3', 'PS4'],                 'Two strong pathogenic'),
    (['PM2', 'PP3'],                 'Absent + computational only'),
    (['BA1'],                        'Common variant (>5% freq)'),
    (['BS1', 'BP4'],                 'Higher freq than expected + in silico benign'),
    (['PVS1', 'PS2', 'PM2'],         'Null + de novo + absent'),
]

print(f'{"Criteria":<30} {"Classification":<20} Description')
print('-' * 85)
for criteria, desc in test_cases:
    result = classify_variant(criteria)
    print(f'{", ".join(criteria):<30} {result:<20} {desc}')

---

## 3. Clinical Databases

Clinical variant interpretation depends on several key public databases.

### 3.1 ClinVar

**ClinVar** (NCBI) is the primary public archive of variant-condition interpretations submitted by clinical and research laboratories.

Key concepts:
- **Submission-based**: Multiple labs can submit interpretations for the same variant
- **Review status**: Stars indicate level of evidence/consensus (0-4 stars)
- **Aggregate interpretation**: ClinVar summarizes conflicting interpretations
- **Accession**: Each variant has a Variation ID; each condition-variant pair has an RCV accession

```
ClinVar Review Status (star rating):

  ****  Practice guideline
  ***   Reviewed by expert panel
  **    Criteria provided, multiple submitters, no conflicts
  *     Criteria provided, single submitter (or conflicting)
  (0)   No assertion criteria provided / no classification
```

### 3.2 OMIM (Online Mendelian Inheritance in Man)

OMIM catalogs known Mendelian disorders and their associated genes. Each entry includes:
- Gene function and structure
- Allelic variants with clinical descriptions
- Gene-phenotype relationships with inheritance patterns
- MIM numbers: `*` = gene, `#` = phenotype with known molecular basis, `%` = phenotype, `+` = gene and phenotype

### 3.3 gnomAD (Genome Aggregation Database)

gnomAD provides population allele frequencies from >140,000 exomes and >76,000 genomes (v2.1.1) / >800,000 individuals (v4). Essential for:
- **Filtering common variants**: Variants with high population frequency are unlikely to cause rare Mendelian disease
- **Population-specific frequencies**: Critical for accurate filtering in diverse populations
- **Constraint metrics**: o/e ratios, pLI scores, LOEUF -- how tolerant is the gene to loss-of-function?

### 3.4 HGMD and LOVD

- **HGMD** (Human Gene Mutation Database): Curated database of germline mutations associated with human inherited disease. Commercial access (public version lags).
- **LOVD** (Leiden Open Variation Database): Open-source, gene-specific variant databases. Many gene-specific databases use LOVD infrastructure.

In [ ]:
# Querying ClinVar via NCBI E-utilities API

def query_clinvar(variant_description: str) -> dict | None:
    """Query ClinVar for a variant using NCBI E-utilities.
    
    Args:
        variant_description: Search term, e.g. 'BRCA1[gene] AND missense'
                            or a specific variant like 'NM_007294.4:c.5266dupC'
    Returns:
        dict with search results or None if requests unavailable.
    """
    if not HAS_REQUESTS:
        return None
    
    base_url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
    
    # Step 1: search for variant IDs
    search_url = f'{base_url}/esearch.fcgi'
    search_params = {
        'db': 'clinvar',
        'term': variant_description,
        'retmax': 5,
        'retmode': 'json',
    }
    resp = requests.get(search_url, params=search_params, timeout=10)
    resp.raise_for_status()
    search_data = resp.json()
    
    id_list = search_data.get('esearchresult', {}).get('idlist', [])
    if not id_list:
        return {'query': variant_description, 'count': 0, 'records': []}
    
    # Step 2: fetch summaries for found IDs
    summary_url = f'{base_url}/esummary.fcgi'
    summary_params = {
        'db': 'clinvar',
        'id': ','.join(id_list),
        'retmode': 'json',
    }
    resp = requests.get(summary_url, params=summary_params, timeout=10)
    resp.raise_for_status()
    summary_data = resp.json()
    
    records = []
    result_block = summary_data.get('result', {})
    for uid in id_list:
        entry = result_block.get(uid, {})
        if entry:
            records.append({
                'uid': uid,
                'title': entry.get('title', ''),
                'clinical_significance': entry.get('clinical_significance', {}).get('description', 'N/A'),
                'gene_sort': entry.get('gene_sort', ''),
                'review_status': entry.get('clinical_significance', {}).get('review_status', 'N/A'),
            })
    
    return {
        'query': variant_description,
        'count': int(search_data['esearchresult'].get('count', 0)),
        'records': records,
    }


# Example: search for pathogenic BRCA1 variants
result = query_clinvar('BRCA1[gene] AND pathogenic[clinical_significance]')

if result is None:
    print('requests library not available -- showing mock example')
    print()
    # Mock data for demonstration
    result = {
        'query': 'BRCA1[gene] AND pathogenic[clinical_significance]',
        'count': 3847,
        'records': [
            {'uid': '17661', 'title': 'NM_007294.4(BRCA1):c.5266dupC (p.Gln1756fs)',
             'clinical_significance': 'Pathogenic', 'gene_sort': 'BRCA1',
             'review_status': 'reviewed by expert panel'},
            {'uid': '37613', 'title': 'NM_007294.4(BRCA1):c.68_69del (p.Glu23fs)',
             'clinical_significance': 'Pathogenic', 'gene_sort': 'BRCA1',
             'review_status': 'reviewed by expert panel'},
            {'uid': '17694', 'title': 'NM_007294.4(BRCA1):c.5096G>A (p.Arg1699Gln)',
             'clinical_significance': 'Pathogenic', 'gene_sort': 'BRCA1',
             'review_status': 'reviewed by expert panel'},
        ],
    }

print(f"Query: {result['query']}")
print(f"Total results: {result['count']}")
print(f"\nShowing first {len(result['records'])} records:")
for rec in result['records']:
    print(f"  [{rec['uid']}] {rec['title']}")
    print(f"           Significance: {rec['clinical_significance']}")
    print(f"           Review: {rec['review_status']}")
    print()

In [ ]:
# Simulating gnomAD-style population frequency filtering

# In clinical practice, variants are filtered by allele frequency to exclude
# common polymorphisms that cannot cause rare Mendelian disease.

GNOMAD_POPULATIONS = [
    'afr',   # African/African-American
    'amr',   # Admixed American / Latino
    'asj',   # Ashkenazi Jewish
    'eas',   # East Asian
    'fin',   # Finnish
    'nfe',   # Non-Finnish European
    'sas',   # South Asian
]

# Example variants with population frequencies
variants = [
    {
        'variant': 'chr17:41245466 G>A (BRCA1 p.Arg1699Gln)',
        'freq': {'afr': 0.0, 'amr': 0.0, 'asj': 0.0, 'eas': 0.0,
                 'fin': 0.0, 'nfe': 0.000003, 'sas': 0.0},
    },
    {
        'variant': 'chr7:117559590 T>G (CFTR p.Phe508del -- actually a deletion)',
        'freq': {'afr': 0.003, 'amr': 0.008, 'asj': 0.012, 'eas': 0.0,
                 'fin': 0.01, 'nfe': 0.012, 'sas': 0.001},
    },
    {
        'variant': 'chr1:55039974 G>A (common benign polymorphism)',
        'freq': {'afr': 0.15, 'amr': 0.12, 'asj': 0.08, 'eas': 0.20,
                 'fin': 0.10, 'nfe': 0.11, 'sas': 0.18},
    },
    {
        'variant': 'chr11:5226774 A>T (HBB p.Glu7Val -- sickle cell)',
        'freq': {'afr': 0.06, 'amr': 0.003, 'asj': 0.0, 'eas': 0.0,
                 'fin': 0.0, 'nfe': 0.0001, 'sas': 0.002},
    },
]


def filter_by_frequency(variants: list[dict], max_af: float = 0.01,
                        max_popmax: float = 0.01) -> tuple[list, list]:
    """Filter variants by population allele frequency.
    
    Args:
        variants: List of variant dicts with 'freq' per population.
        max_af: Maximum global allele frequency threshold.
        max_popmax: Maximum frequency in any single population.
    Returns:
        (rare_variants, filtered_out) -- variants passing and failing the filter.
    """
    rare = []
    filtered_out = []
    for var in variants:
        freqs = var['freq'].values()
        popmax = max(freqs)
        global_af = sum(freqs) / len(freqs)  # simplified; real gnomAD uses weighted average
        if popmax > max_popmax:
            filtered_out.append((var, popmax, 'popmax exceeds threshold'))
        elif global_af > max_af:
            filtered_out.append((var, global_af, 'global AF exceeds threshold'))
        else:
            rare.append(var)
    return rare, filtered_out


rare, filtered = filter_by_frequency(variants, max_popmax=0.05)

print("=== Variants passing frequency filter (popmax < 5%) ===")
for v in rare:
    popmax = max(v['freq'].values())
    print(f"  PASS  {v['variant']}  (popmax={popmax:.6f})")

print("\n=== Variants filtered out ===")
for v, freq, reason in filtered:
    print(f"  FAIL  {v['variant']}  ({reason}: {freq:.4f})")

### 3.5 Gene Constraint Metrics from gnomAD

gnomAD provides constraint metrics that tell us how tolerant a gene is to different types of mutations:

- **pLI (probability of Loss-of-function Intolerance)**: Probability that a gene is intolerant to heterozygous LoF. pLI > 0.9 indicates a haploinsufficient gene.
- **LOEUF (Loss-of-function Observed/Expected Upper bound Fraction)**: Ratio of observed to expected LoF variants. Lower LOEUF = more constrained. LOEUF < 0.35 indicates strong constraint.
- **Missense o/e**: Observed/expected ratio of missense variants. Lower values indicate missense constraint.

```
Gene Constraint Examples:

Gene     pLI    LOEUF   Missense o/e   Interpretation
----     ---    -----   ------------   --------------
BRCA1    0.00   0.58    0.75           LoF tolerated (recessive), mild missense constraint
TP53     0.99   0.08    0.48           Highly LoF-intolerant, strong missense constraint
MECP2    1.00   0.06    0.38           Highly constrained (X-linked dominant)
HBB      0.00   1.15    1.02           LoF tolerated (carriers healthy, recessive disease)
TTN      0.00   0.69    0.97           Large gene, many benign truncations
```

---

## 4. Variant Effect Prediction

In silico prediction tools estimate the functional impact of variants based on sequence conservation, protein structure, and machine learning models.

### 4.1 Key Predictors

**SIFT (Sorting Intolerant From Tolerant)**
- Based on sequence conservation in homologous proteins
- Score 0-1: < 0.05 = "Damaging", >= 0.05 = "Tolerated"
- Simple and fast, but limited to conservation alone

**PolyPhen-2 (Polymorphism Phenotyping v2)**
- Uses sequence conservation + protein structural features
- Score 0-1: > 0.908 = "Probably damaging", 0.446-0.908 = "Possibly damaging", < 0.446 = "Benign"
- Two models: HumDiv (for rare alleles) and HumVar (for clinical diagnostics)

**CADD (Combined Annotation Dependent Depletion)**
- Integrates 60+ genomic features using machine learning
- Phred-scaled score: 10 = top 10% most deleterious, 20 = top 1%, 30 = top 0.1%
- Can score any variant type (coding, non-coding, indels)
- Widely used as a general-purpose deleteriousness score

**REVEL (Rare Exome Variant Ensemble Learner)**
- Ensemble of 13 individual predictors
- Score 0-1: designed specifically for rare missense variants
- Generally better discrimination than individual tools
- Recommended threshold: > 0.5 for likely deleterious, > 0.75 for higher confidence

**AlphaMissense (DeepMind, 2023)**
- Protein language model fine-tuned on pathogenicity data
- Score 0-1: < 0.34 = "Likely Benign", > 0.564 = "Likely Pathogenic"
- Pre-computed for all possible human missense variants (~71 million)
- State-of-the-art performance in benchmarks

### 4.2 Splicing Predictors

Variants near or within splice sites can disrupt mRNA splicing. Specialized tools predict splicing impact:

**SpliceAI (Illumina)**
- Deep learning model predicting cryptic splice site creation/disruption
- Reports delta scores (0-1) for donor gain, donor loss, acceptor gain, acceptor loss
- Can detect deep intronic splicing variants (up to 10 kb from exon)
- Score interpretation: > 0.2 suggests splicing impact, > 0.5 high confidence, > 0.8 very high

### 4.3 Conservation-Based Methods

Cross-species conservation is a fundamental signal for variant interpretation:
- **PhyloP**: Per-base conservation score. Positive = conserved, negative = fast-evolving
- **GERP++**: Identifies constrained elements. RS score > 2 indicates evolutionary constraint
- **PhastCons**: Probability that a site belongs to a conserved element (0-1)

### 4.4 Combining Predictors: Ensemble Approaches

No single predictor is perfect. Best practices:
1. **Use multiple tools** and look for concordance
2. **Meta-predictors** (REVEL, CADD, AlphaMissense) already integrate multiple signals
3. **ACMG criteria PP3/BP4** explicitly reference computational evidence
4. **ClinGen-recommended thresholds** provide calibrated cutoffs for specific tools

In [ ]:
# Simulating in silico predictions for a set of variants

class VariantPrediction(NamedTuple):
    variant: str
    gene: str
    consequence: str
    sift_score: float        # 0-1, lower = more damaging
    polyphen2_score: float   # 0-1, higher = more damaging
    cadd_phred: float        # Phred-scaled, higher = more damaging
    revel_score: float       # 0-1, higher = more damaging
    alphamissense: float     # 0-1, higher = more pathogenic


# Example variants with realistic prediction scores
predictions = [
    VariantPrediction('p.Arg1699Gln', 'BRCA1', 'missense',
                      sift_score=0.0, polyphen2_score=0.999, cadd_phred=29.5,
                      revel_score=0.85, alphamissense=0.92),
    VariantPrediction('p.Val600Glu', 'BRAF', 'missense',
                      sift_score=0.0, polyphen2_score=0.998, cadd_phred=33.0,
                      revel_score=0.95, alphamissense=0.97),
    VariantPrediction('p.Ala222Val', 'MTHFR', 'missense',
                      sift_score=0.08, polyphen2_score=0.482, cadd_phred=18.2,
                      revel_score=0.25, alphamissense=0.15),
    VariantPrediction('p.Asp835Tyr', 'FLT3', 'missense',
                      sift_score=0.01, polyphen2_score=0.91, cadd_phred=26.1,
                      revel_score=0.72, alphamissense=0.78),
    VariantPrediction('p.Pro72Arg', 'TP53', 'missense',
                      sift_score=0.22, polyphen2_score=0.10, cadd_phred=9.8,
                      revel_score=0.12, alphamissense=0.08),
]


def interpret_sift(score: float) -> str:
    return 'Damaging' if score < 0.05 else 'Tolerated'

def interpret_polyphen2(score: float) -> str:
    if score > 0.908:
        return 'Probably damaging'
    elif score > 0.446:
        return 'Possibly damaging'
    return 'Benign'

def interpret_cadd(score: float) -> str:
    if score >= 25:
        return 'Top 0.3% deleterious'
    elif score >= 20:
        return 'Top 1% deleterious'
    elif score >= 15:
        return 'Top 3% deleterious'
    return 'Below suggested cutoff'

def interpret_revel(score: float) -> str:
    if score > 0.75:
        return 'Likely pathogenic'
    elif score > 0.5:
        return 'Possibly pathogenic'
    return 'Likely benign'

def interpret_alphamissense(score: float) -> str:
    if score > 0.564:
        return 'Likely pathogenic'
    elif score < 0.34:
        return 'Likely benign'
    return 'Ambiguous'


print(f'{"Variant":<20} {"Gene":<8} {"SIFT":<14} {"PolyPhen-2":<22} '
      f'{"CADD":<22} {"REVEL":<20} {"AlphaMissense"}')
print('=' * 130)
for p in predictions:
    print(f'{p.variant:<20} {p.gene:<8} '
          f'{p.sift_score:.2f} {interpret_sift(p.sift_score):<10} '
          f'{p.polyphen2_score:.3f} {interpret_polyphen2(p.polyphen2_score):<17} '
          f'{p.cadd_phred:>5.1f} {interpret_cadd(p.cadd_phred):<16} '
          f'{p.revel_score:.2f} {interpret_revel(p.revel_score):<15} '
          f'{p.alphamissense:.2f} {interpret_alphamissense(p.alphamissense)}')

In [ ]:
# Concordance analysis: how often do predictors agree?

def count_damaging(pred: VariantPrediction) -> dict:
    """Count how many predictors flag a variant as damaging."""
    calls = {
        'SIFT': pred.sift_score < 0.05,
        'PolyPhen-2': pred.polyphen2_score > 0.446,
        'CADD': pred.cadd_phred >= 20,
        'REVEL': pred.revel_score > 0.5,
        'AlphaMissense': pred.alphamissense > 0.564,
    }
    return calls


print('Concordance of in silico predictions:')
print(f'{"":<20} {"SIFT":<8} {"PP2":<8} {"CADD":<8} {"REVEL":<8} {"AM":<8} {"Consensus"}')
print('-' * 75)

for pred in predictions:
    calls = count_damaging(pred)
    n_damaging = sum(calls.values())
    n_total = len(calls)
    symbols = ['D' if v else '.' for v in calls.values()]
    consensus = f'{n_damaging}/{n_total} damaging'
    
    # ACMG PP3/BP4 guidance: majority agreement supports pathogenic/benign
    if n_damaging >= 4:
        acmg = '-> PP3 (computational support for pathogenic)'
    elif n_damaging <= 1:
        acmg = '-> BP4 (computational support for benign)'
    else:
        acmg = '-> no strong computational consensus'
    
    print(f'{pred.gene + " " + pred.variant:<20}', 
          '   '.join(f'{s:<4}' for s in symbols),
          f'  {consensus:<16} {acmg}')

---

## 5. Pharmacogenomics

**Pharmacogenomics (PGx)** studies how genetic variation affects drug response. Clinically, PGx testing can:
- Predict adverse drug reactions before they occur
- Guide optimal drug dosing
- Select the right drug for the right patient

### 5.1 Drug-Gene Interactions

Key resources:
- **PharmGKB**: Curated knowledge base of drug-gene-variant associations
- **CPIC (Clinical Pharmacogenetics Implementation Consortium)**: Provides dosing guidelines based on genotype
- **FDA Pharmacogenomic Biomarkers**: List of drugs with PGx information in labeling

### 5.2 CYP450 Polymorphisms

The cytochrome P450 enzyme superfamily metabolizes ~75% of all drugs. Key pharmacogenes:

```
CYP450 Metabolizer Phenotypes:

  Ultra-rapid   ->  Increased enzyme activity  ->  Rapid drug clearance
  Metabolizer       (gene duplications)             (may need higher dose
  (UM)                                               or drug is ineffective)

  Normal         ->  Expected enzyme activity  ->  Standard dosing
  Metabolizer
  (NM / EM)

  Intermediate   ->  Reduced enzyme activity   ->  Slower drug clearance
  Metabolizer        (one reduced-function         (may need dose reduction)
  (IM)                allele)

  Poor           ->  Little/no enzyme activity ->  Very slow clearance
  Metabolizer        (two loss-of-function         (high risk of toxicity
  (PM)                alleles)                      or use alternative drug)
```

| Gene | Key Substrates | Clinical Impact |
|------|---------------|----------------|
| **CYP2D6** | Codeine, tamoxifen, many antidepressants | UMs convert codeine to morphine too fast (toxicity); PMs get no benefit from codeine |
| **CYP2C19** | Clopidogrel, PPIs, some antidepressants | PMs have reduced clopidogrel activation (increased cardiovascular risk) |
| **CYP2C9** | Warfarin, phenytoin, NSAIDs | PMs require lower warfarin doses (bleeding risk) |
| **CYP3A4/5** | Tacrolimus, cyclosporine, many drugs | Variable metabolism affects transplant drug dosing |

In [ ]:
# Pharmacogenomics: CYP2D6 metabolizer status prediction

# CYP2D6 star alleles and their activity scores (simplified)
CYP2D6_ALLELES = {
    '*1':  {'activity': 1.0, 'function': 'Normal'},
    '*2':  {'activity': 1.0, 'function': 'Normal'},
    '*3':  {'activity': 0.0, 'function': 'No function'},
    '*4':  {'activity': 0.0, 'function': 'No function'},
    '*5':  {'activity': 0.0, 'function': 'No function (gene deletion)'},
    '*6':  {'activity': 0.0, 'function': 'No function'},
    '*9':  {'activity': 0.5, 'function': 'Decreased'},
    '*10': {'activity': 0.25, 'function': 'Decreased'},
    '*17': {'activity': 0.5, 'function': 'Decreased'},
    '*29': {'activity': 0.5, 'function': 'Decreased'},
    '*41': {'activity': 0.5, 'function': 'Decreased'},
    '*1x2': {'activity': 2.0, 'function': 'Increased (duplication)'},
    '*2x2': {'activity': 2.0, 'function': 'Increased (duplication)'},
}


def get_metabolizer_status(allele1: str, allele2: str) -> tuple[float, str]:
    """Determine CYP2D6 metabolizer status from diplotype.
    
    Uses CPIC Activity Score system:
    - AS = 0: Poor Metabolizer (PM)
    - AS = 0.25-1.0: Intermediate Metabolizer (IM) 
    - AS = 1.25-2.25: Normal Metabolizer (NM)
    - AS > 2.25: Ultrarapid Metabolizer (UM)
    """
    a1 = CYP2D6_ALLELES.get(allele1)
    a2 = CYP2D6_ALLELES.get(allele2)
    if a1 is None or a2 is None:
        raise ValueError(f"Unknown allele: {allele1 if a1 is None else allele2}")
    
    activity_score = a1['activity'] + a2['activity']
    
    if activity_score == 0:
        phenotype = 'Poor Metabolizer (PM)'
    elif activity_score <= 1.0:
        phenotype = 'Intermediate Metabolizer (IM)'
    elif activity_score <= 2.25:
        phenotype = 'Normal Metabolizer (NM)'
    else:
        phenotype = 'Ultrarapid Metabolizer (UM)'
    
    return activity_score, phenotype


# Clinical examples
test_diplotypes = [
    ('*1', '*1',   'Most common genotype'),
    ('*1', '*4',   'Carrier of non-functional allele'),
    ('*4', '*4',   'Homozygous non-functional'),
    ('*1', '*41',  'One decreased function allele'),
    ('*4', '*5',   'Compound het non-functional'),
    ('*1x2', '*1', 'Gene duplication'),
    ('*10', '*10', 'Common in East Asian populations'),
    ('*1', '*10',  'Heterozygous decreased function'),
]

print(f'{"Diplotype":<18} {"Activity Score":<18} {"Phenotype":<32} Note')
print('=' * 100)
for a1, a2, note in test_diplotypes:
    score, pheno = get_metabolizer_status(a1, a2)
    print(f'{a1}/{a2:<14} {score:<18.2f} {pheno:<32} {note}')

In [ ]:
# CPIC dosing recommendations for codeine based on CYP2D6 metabolizer status

CPIC_CODEINE_GUIDELINES = {
    'Ultrarapid Metabolizer (UM)': {
        'recommendation': 'AVOID codeine. Use alternative analgesic (e.g., non-tramadol, non-codeine).',
        'risk': 'HIGH - Rapid conversion to morphine; risk of life-threatening toxicity.',
        'evidence': 'Strong',
    },
    'Normal Metabolizer (NM)': {
        'recommendation': 'Use codeine per standard dosing guidelines.',
        'risk': 'Normal - Expected analgesic response.',
        'evidence': 'Strong',
    },
    'Intermediate Metabolizer (IM)': {
        'recommendation': 'Use codeine per standard dosing guidelines. '
                          'Monitor for reduced efficacy; consider alternative if inadequate response.',
        'risk': 'LOW - Reduced (but not absent) conversion to morphine.',
        'evidence': 'Moderate',
    },
    'Poor Metabolizer (PM)': {
        'recommendation': 'AVOID codeine. Use alternative analgesic (non-tramadol, non-codeine).',
        'risk': 'No therapeutic effect expected - minimal conversion to morphine.',
        'evidence': 'Strong',
    },
}

print('CPIC Dosing Guidelines: Codeine and CYP2D6')
print('=' * 80)
for phenotype, info in CPIC_CODEINE_GUIDELINES.items():
    print(f'\n{phenotype}:')
    print(f'  Recommendation: {info["recommendation"]}')
    print(f'  Risk level:     {info["risk"]}')
    print(f'  Evidence:       {info["evidence"]}')

---

## 6. Cancer Genomics Basics

Cancer genomics applies sequencing to understand tumor biology and guide treatment. Key concepts differ from germline genetics.

### 6.1 Somatic vs Germline Variants

```
Germline vs Somatic Variants:

  GERMLINE                          SOMATIC
  - Present in every cell           - Present only in tumor cells
  - Inherited from parents          - Acquired during lifetime
  - Detected in blood/saliva        - Detected by tumor-normal comparison
  - Heterozygous (50%) or           - Variable allele fraction (VAF):
    homozygous (100%) VAF              often 5-50%, reflects clonality
  - Hereditary cancer risk          - Guides treatment of current tumor
    (e.g., BRCA1/2)                   (e.g., EGFR mutations in NSCLC)
```

### 6.2 Driver vs Passenger Mutations

- **Driver mutations**: Confer growth advantage, causally implicated in cancer. Found in known oncogenes (BRAF, KRAS, EGFR) or tumor suppressors (TP53, RB1, APC).
- **Passenger mutations**: Neutral mutations present because they occurred in a cell that later clonally expanded. Vastly outnumber drivers.

Distinguishing drivers from passengers:
- Recurrence across tumors (same gene/position mutated in many patients)
- Functional impact (activating mutations in oncogenes, loss-of-function in tumor suppressors)
- Hotspot analysis (e.g., BRAF V600E, KRAS G12D)
- Statistical methods (MutSig, dNdScv) that compare observed vs expected mutation rates

### 6.3 Tumor Mutational Burden (TMB)

**TMB** is the total number of somatic mutations per megabase (mut/Mb) of coding region in a tumor. It serves as a biomarker for immunotherapy response:

- **High TMB** (typically >= 10 mut/Mb): More neoantigens, better response to immune checkpoint inhibitors (pembrolizumab, nivolumab)
- **Low TMB**: Fewer neoantigens, less likely to respond to immunotherapy alone

TMB varies widely by cancer type:
- Melanoma: ~10-50 mut/Mb (UV damage)
- Lung cancer (smokers): ~10-30 mut/Mb (tobacco carcinogens)
- Pediatric tumors: ~0.5-2 mut/Mb (fewer cell divisions)
- Microsatellite unstable tumors: very high TMB

### 6.4 Microsatellite Instability (MSI)

**MSI** results from deficient DNA mismatch repair (dMMR). It is:
- Detected by comparing microsatellite repeat lengths between tumor and normal
- **MSI-H (high)**: Unstable at >= 2 of 5 standard markers (Bethesda panel), or >= 30% of markers
- A pan-cancer biomarker for immunotherapy: FDA approved pembrolizumab for any MSI-H solid tumor
- Most common in colorectal cancer (15%), endometrial cancer (20-30%)
- Can be hereditary (Lynch syndrome: MLH1, MSH2, MSH6, PMS2 germline mutations)

### 6.5 Key Cancer Genomics Resources

- **OncoKB** (Memorial Sloan Kettering): Curated database of oncogenic variants and their clinical actionability. Levels of evidence from 1 (FDA-approved) to 4 (biological evidence only).
- **cBioPortal**: Open platform for exploring multidimensional cancer genomics data (TCGA, AACR Project GENIE, etc.)
- **COSMIC**: Catalogue of Somatic Mutations in Cancer

In [ ]:
# Cancer genomics: TMB calculation from VCF-like data

def calculate_tmb(somatic_variants: list[dict], coding_region_mb: float = 30.0) -> float:
    """Calculate Tumor Mutational Burden.
    
    Args:
        somatic_variants: List of somatic variant dicts with 'type' and 'vaf' keys.
        coding_region_mb: Size of coding region assessed (Mb). WES ~ 30-35 Mb,
                         panels vary (e.g., FoundationOne CDx ~ 0.8 Mb).
    Returns:
        TMB in mutations per megabase.
    """
    # Count coding, non-synonymous somatic variants
    coding_mutations = [
        v for v in somatic_variants 
        if v['type'] in ('missense', 'nonsense', 'frameshift', 'splice_site', 'inframe_indel')
    ]
    return len(coding_mutations) / coding_region_mb


# Simulated somatic variants for three different tumor samples
import random
random.seed(42)

variant_types = ['missense', 'nonsense', 'frameshift', 'splice_site',
                 'synonymous', 'intronic', 'inframe_indel']
coding_types = {'missense', 'nonsense', 'frameshift', 'splice_site', 'inframe_indel'}

tumors = {
    'Melanoma (UV-exposed)': [
        {'type': random.choice(variant_types), 'vaf': random.uniform(0.05, 0.6)}
        for _ in range(850)  # high mutation count
    ],
    'Lung adenocarcinoma (smoker)': [
        {'type': random.choice(variant_types), 'vaf': random.uniform(0.05, 0.5)}
        for _ in range(450)
    ],
    'Pediatric glioma': [
        {'type': random.choice(variant_types), 'vaf': random.uniform(0.1, 0.5)}
        for _ in range(40)  # low mutation count
    ],
}

print(f'{"Tumor Type":<35} {"Total Variants":<18} {"Coding":<10} {"TMB (mut/Mb)":<15} Interpretation')
print('=' * 110)
for tumor_name, variants in tumors.items():
    coding_count = sum(1 for v in variants if v['type'] in coding_types)
    tmb = calculate_tmb(variants, coding_region_mb=30.0)
    
    if tmb >= 10:
        interp = 'TMB-High -> consider immunotherapy'
    elif tmb >= 6:
        interp = 'TMB-Intermediate'
    else:
        interp = 'TMB-Low'
    
    print(f'{tumor_name:<35} {len(variants):<18} {coding_count:<10} {tmb:<15.1f} {interp}')

In [ ]:
# Somatic variant interpretation: oncogenic driver identification

# Known cancer hotspot mutations (simplified OncoKB-like database)
CANCER_HOTSPOTS = {
    ('BRAF', 'V600E'): {
        'oncogenicity': 'Oncogenic',
        'level': '1 (FDA-approved therapy)',
        'therapies': ['Vemurafenib', 'Dabrafenib + Trametinib'],
        'cancer_types': ['Melanoma', 'NSCLC', 'Colorectal (with cetuximab)'],
    },
    ('EGFR', 'L858R'): {
        'oncogenicity': 'Oncogenic',
        'level': '1 (FDA-approved therapy)',
        'therapies': ['Osimertinib', 'Erlotinib', 'Gefitinib'],
        'cancer_types': ['NSCLC'],
    },
    ('EGFR', 'T790M'): {
        'oncogenicity': 'Oncogenic (resistance)',
        'level': '1 (FDA-approved therapy)',
        'therapies': ['Osimertinib (3rd-gen TKI)'],
        'cancer_types': ['NSCLC (acquired resistance to 1st/2nd-gen EGFR TKIs)'],
    },
    ('KRAS', 'G12C'): {
        'oncogenicity': 'Oncogenic',
        'level': '1 (FDA-approved therapy)',
        'therapies': ['Sotorasib', 'Adagrasib'],
        'cancer_types': ['NSCLC'],
    },
    ('KRAS', 'G12D'): {
        'oncogenicity': 'Oncogenic',
        'level': '3B (clinical evidence in other tumor types)',
        'therapies': ['MRTX1133 (investigational)'],
        'cancer_types': ['Pancreatic', 'Colorectal'],
    },
    ('PIK3CA', 'H1047R'): {
        'oncogenicity': 'Oncogenic',
        'level': '1 (FDA-approved therapy)',
        'therapies': ['Alpelisib (with fulvestrant)'],
        'cancer_types': ['Breast cancer (HR+/HER2-)'],
    },
}


# Simulated tumor profiling results
patient_variants = [
    {'gene': 'BRAF', 'mutation': 'V600E', 'vaf': 0.42},
    {'gene': 'TP53', 'mutation': 'R175H', 'vaf': 0.65},
    {'gene': 'EGFR', 'mutation': 'L858R', 'vaf': 0.35},
    {'gene': 'KRAS', 'mutation': 'G12C', 'vaf': 0.28},
    {'gene': 'MYC', 'mutation': 'A123T', 'vaf': 0.15},  # Unknown significance
]

print('Tumor Profiling Report: Somatic Variant Interpretation')
print('=' * 80)
for var in patient_variants:
    key = (var['gene'], var['mutation'])
    hotspot = CANCER_HOTSPOTS.get(key)
    
    print(f"\n{var['gene']} {var['mutation']} (VAF: {var['vaf']:.0%})")
    if hotspot:
        print(f"  Oncogenicity:     {hotspot['oncogenicity']}")
        print(f"  Evidence level:   {hotspot['level']}")
        print(f"  Therapies:        {', '.join(hotspot['therapies'])}")
        print(f"  Cancer types:     {', '.join(hotspot['cancer_types'])}")
    else:
        print(f"  Not found in hotspot database -- Variant of Unknown Significance")
        print(f"  Recommendation: Check OncoKB, COSMIC, literature for evidence")

---

## 7. Polygenic Risk Scores (PRS)

While Mendelian genetics focuses on single high-impact variants, most common diseases (type 2 diabetes, coronary artery disease, schizophrenia) are influenced by thousands of variants, each with a small effect.

### 7.1 From Single Variants to Genome-Wide Risk

```
Genetic Architecture Spectrum:

  Mendelian             Oligogenic              Complex/Polygenic
  (1 gene)             (few genes)             (many genes)
  
  |------|              |------|                |------|
  Large effect          Moderate effects        Small effects
  (OR > 5)              (OR 2-5)                (OR 1.01-1.5)
  Rare variants         Uncommon variants       Common variants
  
  Examples:             Examples:               Examples:
  - Huntington's        - BRCA1/2 breast ca     - T2D (>400 loci)
  - Cystic fibrosis     - Macular degeneration  - CAD (>160 loci)
  - Sickle cell         - Some autism genes      - Height (>10,000 loci)
```

### 7.2 PRS Calculation

A PRS aggregates the effects of many variants into a single score:

**PRS = sum over all SNPs i of: (beta_i * dosage_i)**

Where:
- **beta_i** = effect size (log odds ratio) from GWAS
- **dosage_i** = number of risk alleles (0, 1, or 2) at SNP i

The raw PRS is typically standardized (z-score) against a reference population.

In [ ]:
# Polygenic Risk Score calculation demonstration

# Simplified PRS for coronary artery disease (CAD)
# In reality, a CAD PRS includes >1 million variants; we show the concept with 15

# Each entry: (SNP rsid, risk allele, beta from GWAS, allele frequency)
CAD_PRS_SNPS = [
    ('rs4977574', 'G', 0.29, 0.46),   # 9p21.3 (CDKN2B-AS1)
    ('rs11206510', 'T', 0.15, 0.82),   # 1p32.3 (PCSK9)
    ('rs646776', 'T', 0.18, 0.78),     # 1p13.3 (SORT1)
    ('rs17114036', 'A', 0.19, 0.91),   # 1p32.2 (PPAP2B)
    ('rs9982601', 'T', 0.18, 0.87),    # 21q22.11 (SLC5A3)
    ('rs1333049', 'C', 0.25, 0.47),    # 9p21.3
    ('rs2505083', 'T', 0.10, 0.42),    # 10q24.32
    ('rs12413409', 'G', 0.14, 0.89),   # 10q11.21 (CYP17A1)
    ('rs3184504', 'T', 0.13, 0.49),    # 12q24.12 (SH2B3)
    ('rs964184', 'G', 0.13, 0.13),     # 11q23.3 (ZPR1)
    ('rs579459', 'C', 0.11, 0.21),     # 9q34.2 (ABO)
    ('rs6725887', 'C', 0.17, 0.14),    # 2q33.1 (WDR12)
    ('rs12190287', 'C', 0.12, 0.62),   # 6q23.2 (TCF21)
    ('rs216172', 'G', 0.08, 0.37),     # 7q32.2 (ZC3HC1)
    ('rs17465637', 'C', 0.10, 0.74),   # 1q41 (MIA3)
]


def calculate_prs(snps: list[tuple], genotypes: list[int]) -> float:
    """Calculate a raw Polygenic Risk Score.
    
    Args:
        snps: List of (rsid, risk_allele, beta, allele_freq) tuples.
        genotypes: List of risk allele dosages (0, 1, or 2) for each SNP.
    Returns:
        Raw PRS (sum of beta * dosage).
    """
    if len(snps) != len(genotypes):
        raise ValueError("SNP list and genotype list must have same length")
    return sum(beta * dosage for (_, _, beta, _), dosage in zip(snps, genotypes))


def standardize_prs(raw_prs: float, population_mean: float, population_sd: float) -> float:
    """Convert raw PRS to a z-score relative to population distribution."""
    return (raw_prs - population_mean) / population_sd


def prs_to_percentile(z_score: float) -> float:
    """Convert z-score to population percentile using normal CDF approximation."""
    return 0.5 * (1 + math.erf(z_score / math.sqrt(2))) * 100


# Calculate population mean and SD (expected PRS under HWE)
pop_mean = sum(2 * beta * freq for _, _, beta, freq in CAD_PRS_SNPS)
pop_var = sum((2 * freq * (1 - freq)) * (beta ** 2) for _, _, beta, freq in CAD_PRS_SNPS)
pop_sd = math.sqrt(pop_var)

print(f'PRS model: {len(CAD_PRS_SNPS)} SNPs for coronary artery disease')
print(f'Population mean PRS: {pop_mean:.4f}')
print(f'Population SD:       {pop_sd:.4f}')
print()

# Simulate 5 individuals with different genetic risk profiles
random.seed(123)
individuals = {
    'Person A (low risk)':  [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
    'Person B (average)':   [1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1],
    'Person C (above avg)': [2, 1, 1, 1, 1, 2, 1, 0, 1, 0, 0, 1, 1, 1, 1],
    'Person D (high risk)': [2, 2, 2, 1, 2, 2, 1, 1, 2, 1, 1, 1, 2, 1, 2],
    'Person E (moderate)':  [1, 0, 2, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1],
}

print(f'{"Individual":<25} {"Raw PRS":<12} {"Z-score":<12} {"Percentile":<12} Risk Category')
print('=' * 80)
for name, genotypes in individuals.items():
    raw = calculate_prs(CAD_PRS_SNPS, genotypes)
    z = standardize_prs(raw, pop_mean, pop_sd)
    pct = prs_to_percentile(z)
    
    if pct >= 95:
        category = 'Very high risk (top 5%)'
    elif pct >= 80:
        category = 'High risk (top 20%)'
    elif pct >= 50:
        category = 'Average risk'
    else:
        category = 'Below average risk'
    
    print(f'{name:<25} {raw:<12.4f} {z:<12.2f} {pct:<12.1f} {category}')

### 7.3 PRS: Clinical Utility and Limitations

**Current clinical applications:**
- Breast cancer: PRS can refine risk for women with intermediate risk (e.g., combined with family history and clinical models)
- Coronary artery disease: PRS adds to traditional risk factors (cholesterol, blood pressure, smoking)
- Statin benefit: Individuals with high genetic risk for CAD benefit more from statins

**Key limitations:**
1. **Population specificity**: Most GWAS are from European populations. PRS performance degrades in other ancestries.
2. **Environment not captured**: PRS does not account for lifestyle, diet, or environmental exposures
3. **Modest discrimination**: AUC improvement over clinical risk factors is often small (0.5-5%)
4. **Interpretation challenges**: A high PRS is probabilistic, not deterministic. A person with top 5% PRS may never develop disease.
5. **Equity concerns**: If PRS only works well in European populations, it risks widening health disparities

---

## 8. Ethical Considerations in Clinical Genomics

Genomic medicine raises unique ethical challenges that bioinformaticians must understand.

### 8.1 Incidental (Secondary) Findings

When sequencing a patient's exome/genome for one indication, labs may discover variants unrelated to the primary indication but medically actionable.

**ACMG Secondary Findings List (SF v3.2, 2023):**
- 81 genes associated with highly penetrant, medically actionable conditions
- Examples: BRCA1/2 (cancer risk), LDLR (familial hypercholesterolemia), RYR1 (malignant hyperthermia), KCNQ1/KCNH2 (long QT syndrome)
- Labs are recommended to actively look for and report pathogenic/likely pathogenic variants in these genes
- Patients can opt out of receiving secondary findings

### 8.2 Genetic Discrimination

Genetic information could be used to discriminate in employment, insurance, or other areas.

**Legal protections:**
- **GINA (USA, 2008)**: Genetic Information Nondiscrimination Act -- prohibits genetic discrimination in health insurance and employment. Does NOT cover life, disability, or long-term care insurance.
- **EU GDPR**: Genetic data is classified as sensitive personal data with strict processing requirements.
- **Country-specific laws**: Vary widely; some countries have stronger protections than others.

### 8.3 Data Privacy in Genomics

Genomic data presents unique privacy challenges:
- **Re-identifiability**: Even "anonymized" genomic data can potentially identify individuals (demonstrated by Gymrek et al., 2013 using Y-STRs + surname databases)
- **Familial implications**: One person's genome reveals information about their relatives who have not consented
- **Data sharing tension**: Research benefits from data sharing, but privacy must be protected
- **Solutions**: Controlled-access repositories (dbGaP), federated analysis, differential privacy, consent models

### 8.4 Return of Results

Key questions in returning genomic results:
1. Should VUS be returned to patients? (Generally no in clinical settings, but varies)
2. Should results for untreatable conditions be returned? (e.g., Alzheimer's risk)
3. How should results be communicated? (Genetic counseling is essential)
4. What about pediatric testing? (Testing minors for adult-onset conditions is generally discouraged)
5. What duty do labs have to recontact patients when classifications change?

In [ ]:
# Simulating a clinical genomics report: putting it all together

def generate_clinical_report(patient_id: str, variants: list[dict]) -> str:
    """Generate a simplified clinical genomics report.
    
    Each variant dict has keys:
        gene, hgvs_c, hgvs_p, consequence, zygosity, acmg_criteria,
        gnomad_af, clinvar_significance
    """
    lines = []
    lines.append('=' * 70)
    lines.append('CLINICAL GENOMIC SEQUENCING REPORT')
    lines.append('=' * 70)
    lines.append(f'Patient ID: {patient_id}')
    lines.append(f'Test: Clinical Exome Sequencing')
    lines.append(f'Indication: Suspected hereditary condition')
    lines.append('')
    
    # Classify and sort by clinical significance
    classified = []
    for v in variants:
        acmg_class = classify_variant(v['acmg_criteria'])
        classified.append({**v, 'classification': acmg_class})
    
    # Sort: Pathogenic first, then LP, VUS, LB, Benign
    order = {'Pathogenic': 0, 'Likely Pathogenic': 1, 'VUS': 2,
             'Likely Benign': 3, 'Benign': 4}
    classified.sort(key=lambda v: order.get(v['classification'], 5))
    
    # Reportable findings
    reportable = [v for v in classified if v['classification'] in ('Pathogenic', 'Likely Pathogenic')]
    vus_list = [v for v in classified if v['classification'] == 'VUS']
    
    lines.append('--- REPORTABLE FINDINGS ---')
    if reportable:
        for v in reportable:
            lines.append(f"\n  Gene: {v['gene']}")
            lines.append(f"  Variant: {v['hgvs_c']} ({v['hgvs_p']})")
            lines.append(f"  Consequence: {v['consequence']}")
            lines.append(f"  Zygosity: {v['zygosity']}")
            lines.append(f"  Classification: {v['classification']}")
            lines.append(f"  ACMG criteria: {', '.join(v['acmg_criteria'])}")
            lines.append(f"  gnomAD AF: {v['gnomad_af']}")
            lines.append(f"  ClinVar: {v['clinvar_significance']}")
    else:
        lines.append('  No pathogenic or likely pathogenic variants identified.')
    
    lines.append('\n--- VARIANTS OF UNCERTAIN SIGNIFICANCE ---')
    if vus_list:
        for v in vus_list:
            lines.append(f"  {v['gene']} {v['hgvs_c']} ({v['hgvs_p']}) - "
                         f"{v['consequence']}, {v['zygosity']}")
            lines.append(f"    ACMG criteria: {', '.join(v['acmg_criteria'])}")
    else:
        lines.append('  No VUS to report.')
    
    lines.append('\n--- METHODOLOGY ---')
    lines.append('  Clinical exome sequencing with analysis of ~20,000 genes.')
    lines.append('  Variants classified per ACMG/AMP 2015 guidelines.')
    lines.append('  This test does not detect large CNVs, repeat expansions, or epigenetic changes.')
    lines.append('')
    lines.append('=' * 70)
    
    return '\n'.join(lines)


# Example patient with multiple findings
patient_variants = [
    {
        'gene': 'BRCA2', 'hgvs_c': 'c.5946delT', 'hgvs_p': 'p.Ser1982fs',
        'consequence': 'frameshift', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PVS1', 'PM2', 'PP5'],
        'gnomad_af': '0.000004', 'clinvar_significance': 'Pathogenic',
    },
    {
        'gene': 'CHEK2', 'hgvs_c': 'c.1100delC', 'hgvs_p': 'p.Thr367fs',
        'consequence': 'frameshift', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PVS1', 'PS4', 'PM2'],
        'gnomad_af': '0.002', 'clinvar_significance': 'Pathogenic',
    },
    {
        'gene': 'ATM', 'hgvs_c': 'c.7271T>G', 'hgvs_p': 'p.Val2424Gly',
        'consequence': 'missense', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PM2', 'PP3'],
        'gnomad_af': '0.00006', 'clinvar_significance': 'Uncertain significance',
    },
    {
        'gene': 'MSH6', 'hgvs_c': 'c.3261dupC', 'hgvs_p': 'p.Phe1088fs',
        'consequence': 'frameshift', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PVS1', 'PM2'],
        'gnomad_af': 'absent', 'clinvar_significance': 'Likely pathogenic',
    },
]

report = generate_clinical_report('CG-2026-001', patient_variants)
print(report)

---

## Exercises

### Exercise 1: ACMG Variant Classification

Classify the following variants using the `classify_variant()` function. For each, explain whether the classification makes clinical sense.

1. A nonsense variant in *CFTR* that is absent from gnomAD, reported as pathogenic in ClinVar by multiple labs, and co-segregates with cystic fibrosis in a family. **Criteria: PVS1, PM2, PS4, PP1, PP5**

2. A missense variant in *SCN5A* with conflicting in silico predictions, found at 0.3% frequency in the Finnish population, and seen once in a patient with Brugada syndrome. **Criteria: PP3, PP4**

3. A splice donor variant in *MLH1* found in a patient with colorectal cancer at age 30, absent from gnomAD, predicted to disrupt splicing by SpliceAI (delta = 0.95). **Criteria: PVS1, PM2, PP3, PP4**

4. A synonymous variant in *TP53* found at 12% allele frequency in gnomAD, with no predicted splice effect. **Criteria: BA1, BP7**

In [ ]:
# Exercise 1: Classify these variants
# Use the classify_variant() function defined above.

exercise_variants = [
    ('CFTR nonsense',     ['PVS1', 'PM2', 'PS4', 'PP1', 'PP5']),
    ('SCN5A missense',    ['PP3', 'PP4']),
    ('MLH1 splice',       ['PVS1', 'PM2', 'PP3', 'PP4']),
    ('TP53 synonymous',   ['BA1', 'BP7']),
]

# YOUR CODE HERE
# For each variant, call classify_variant() and print the result.
# Then write a brief comment on whether the classification is clinically reasonable.


### Exercise 2: Population Frequency Filtering

You are analyzing an exome for a patient with a rare autosomal recessive disorder (estimated prevalence 1 in 100,000).

1. What is the maximum expected carrier frequency for the causal variant? (Use Hardy-Weinberg: if disease prevalence = q^2, then carrier frequency = 2pq ~ 2q)
2. Given this, what gnomAD allele frequency cutoff would you use?
3. Filter the following variants and explain which ones could potentially be causal:

```
Variant A: gnomAD AF = 0.15        (15%)
Variant B: gnomAD AF = 0.003       (0.3%)
Variant C: gnomAD AF = 0.0001      (0.01%)
Variant D: absent from gnomAD
Variant E: gnomAD AF = 0.05        (5%)
```

In [ ]:
# Exercise 2: Population frequency filtering

# Step 1: Calculate expected carrier frequency
disease_prevalence = 1 / 100_000  # q^2 for autosomal recessive

# YOUR CODE HERE
# q = sqrt(disease_prevalence)
# carrier_frequency = 2 * p * q ≈ 2 * q (since p ≈ 1 for rare diseases)
# Then determine a reasonable AF cutoff and filter the variants.


### Exercise 3: In Silico Predictor Evaluation

Given the following variant predictions, determine the ACMG computational evidence code (PP3 or BP4) for each:

| Variant | SIFT | PolyPhen-2 | CADD | REVEL | AlphaMissense |
|---------|------|-----------|------|-------|---------------|
| p.Gly12Asp | 0.00 | 0.999 | 32.0 | 0.91 | 0.95 |
| p.Ala45Val | 0.35 | 0.05 | 8.2 | 0.08 | 0.12 |
| p.Arg233Trp | 0.01 | 0.85 | 22.5 | 0.55 | 0.48 |
| p.Leu567Pro | 0.00 | 0.98 | 27.8 | 0.82 | 0.89 |

For each variant:
1. Interpret each individual predictor score
2. Determine the consensus (how many call it damaging?)
3. Assign PP3 (supports pathogenic), BP4 (supports benign), or neither

In [ ]:
# Exercise 3: Evaluate in silico predictors

exercise_predictions = [
    VariantPrediction('p.Gly12Asp', 'KRAS', 'missense', 0.00, 0.999, 32.0, 0.91, 0.95),
    VariantPrediction('p.Ala45Val', 'UNKNOWN', 'missense', 0.35, 0.05, 8.2, 0.08, 0.12),
    VariantPrediction('p.Arg233Trp', 'UNKNOWN', 'missense', 0.01, 0.85, 22.5, 0.55, 0.48),
    VariantPrediction('p.Leu567Pro', 'UNKNOWN', 'missense', 0.00, 0.98, 27.8, 0.82, 0.89),
]

# YOUR CODE HERE
# Use the count_damaging() function and the interpretation functions
# to evaluate each variant.


### Exercise 4: Pharmacogenomics Case Study

A 55-year-old patient is prescribed **codeine** for pain management after surgery. Their pharmacogenomic test reveals:
- CYP2D6 diplotype: `*1/*4`

1. Determine the patient's CYP2D6 activity score and metabolizer phenotype
2. What is the clinical recommendation for codeine?
3. If the patient's diplotype were `*1x2/*1`, how would the recommendation change?
4. Why is CYP2D6 genotyping important specifically for codeine (hint: think about the drug's mechanism)?

In [ ]:
# Exercise 4: Pharmacogenomics case study

# YOUR CODE HERE
# 1. Use get_metabolizer_status() to determine the phenotype for *1/*4
# 2. Look up the CPIC recommendation in CPIC_CODEINE_GUIDELINES
# 3. Repeat for *1x2/*1
# 4. Answer the question about codeine's mechanism in a comment or print statement


### Exercise 5: Clinical Genomics Case -- Integrative Analysis

A 28-year-old woman presents with early-onset breast cancer. Clinical exome sequencing reveals the following variants. For each variant, determine:
1. ACMG classification
2. Clinical actionability
3. Whether it should be reported as a primary finding, secondary finding, or VUS

**Variant 1**: *BRCA1* c.5266dupC (p.Gln1756fs)
- Consequence: frameshift
- gnomAD: absent
- ClinVar: Pathogenic (reviewed by expert panel)
- ACMG criteria to apply: PVS1, PS4, PM2, PP5

**Variant 2**: *PALB2* c.2167G>A (p.Gly723Arg)
- Consequence: missense
- gnomAD AF: 0.00008
- ClinVar: Uncertain significance
- In silico: SIFT=0.02, PolyPhen-2=0.75, CADD=23.5, REVEL=0.58
- ACMG criteria to apply: PM2, PP3

**Variant 3**: *LDLR* c.1187-10G>A
- Consequence: intronic (near splice site)
- gnomAD AF: 0.0002
- ClinVar: Likely pathogenic
- SpliceAI delta score: 0.85 (acceptor loss)
- ACMG criteria to apply: PVS1, PM2, PP3, PP5
- Note: LDLR is on the ACMG Secondary Findings list

**Variant 4**: *MTHFR* c.665C>T (p.Ala222Val)
- Consequence: missense
- gnomAD AF: 0.32
- ClinVar: Benign/Likely benign
- ACMG criteria to apply: BA1

In [ ]:
# Exercise 5: Integrative clinical analysis

case_variants = [
    {
        'gene': 'BRCA1', 'hgvs_c': 'c.5266dupC', 'hgvs_p': 'p.Gln1756fs',
        'consequence': 'frameshift', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PVS1', 'PS4', 'PM2', 'PP5'],
        'gnomad_af': 'absent', 'clinvar_significance': 'Pathogenic',
    },
    {
        'gene': 'PALB2', 'hgvs_c': 'c.2167G>A', 'hgvs_p': 'p.Gly723Arg',
        'consequence': 'missense', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PM2', 'PP3'],
        'gnomad_af': '0.00008', 'clinvar_significance': 'Uncertain significance',
    },
    {
        'gene': 'LDLR', 'hgvs_c': 'c.1187-10G>A', 'hgvs_p': 'splice region',
        'consequence': 'splice_region', 'zygosity': 'heterozygous',
        'acmg_criteria': ['PVS1', 'PM2', 'PP3', 'PP5'],
        'gnomad_af': '0.0002', 'clinvar_significance': 'Likely pathogenic',
    },
    {
        'gene': 'MTHFR', 'hgvs_c': 'c.665C>T', 'hgvs_p': 'p.Ala222Val',
        'consequence': 'missense', 'zygosity': 'heterozygous',
        'acmg_criteria': ['BA1'],
        'gnomad_af': '0.32', 'clinvar_significance': 'Benign/Likely benign',
    },
]

# YOUR CODE HERE
# 1. Classify each variant using classify_variant()
# 2. Determine: is it a primary finding, secondary finding, or VUS?
# 3. What clinical actions should follow for each reportable finding?
# 
# Hint: BRCA1 is directly related to the patient's breast cancer (primary finding).
# LDLR is on the ACMG SF list (secondary finding for familial hypercholesterolemia).


### Exercise 6: TMB and MSI Interpretation

A colorectal cancer patient has the following tumor profiling results:
- 380 somatic coding mutations detected across 1.1 Mb of panel sequencing
- 4 out of 5 Bethesda microsatellite markers are unstable

1. Calculate the TMB. Is this TMB-high or TMB-low?
2. What is the MSI status? (MSI-H requires >= 2 of 5 markers unstable)
3. Based on TMB + MSI status, is this patient a candidate for immune checkpoint inhibitors?
4. What germline genes should be tested given the MSI-H finding? (Think Lynch syndrome)

In [ ]:
# Exercise 6: TMB and MSI interpretation

# Given data
coding_mutations = 380
panel_size_mb = 1.1
unstable_markers = 4
total_markers = 5

# YOUR CODE HERE
# 1. Calculate TMB = coding_mutations / panel_size_mb
# 2. Determine MSI status
# 3. Assess immunotherapy candidacy
# 4. List Lynch syndrome genes (MLH1, MSH2, MSH6, PMS2, EPCAM)


---

## Summary

This module covered the core concepts of clinical genomics:

| Topic | Key Takeaway |
|-------|-------------|
| **Precision medicine** | Genomics increasingly guides diagnosis, treatment, and prevention |
| **ACMG classification** | Standardized 5-tier system with evidence-based criteria codes |
| **Clinical databases** | ClinVar, OMIM, gnomAD are essential for variant interpretation |
| **In silico predictors** | Use multiple tools; meta-predictors (REVEL, CADD, AlphaMissense) perform best |
| **Pharmacogenomics** | CYP450 genotype guides drug selection and dosing (CPIC guidelines) |
| **Cancer genomics** | Somatic profiling identifies drivers; TMB and MSI guide immunotherapy |
| **Polygenic risk scores** | Aggregate many small-effect variants; promising but limited by population bias |
| **Ethics** | Incidental findings, discrimination, privacy, and return of results are ongoing challenges |

### Further Reading
- Richards et al. (2015). "Standards and guidelines for the interpretation of sequence variants." *Genetics in Medicine*. 17(5):405-424.
- Relling & Klein (2011). "CPIC: Clinical Pharmacogenetics Implementation Consortium." *Clinical Pharmacology & Therapeutics*. 89(3):464-467.
- Cheng et al. (2023). "Accurate proteome-wide missense variant effect prediction with AlphaMissense." *Science*. 381(6664).
- Khera et al. (2018). "Genome-wide polygenic scores for common diseases identify individuals with risk equivalent to monogenic mutations." *Nature Genetics*. 50:1219-1224.
- ClinGen: https://clinicalgenome.org/
- ClinVar: https://www.ncbi.nlm.nih.gov/clinvar/
- gnomAD: https://gnomad.broadinstitute.org/
- PharmGKB: https://www.pharmgkb.org/
- OncoKB: https://www.oncokb.org/

---

[← Previous: Deep Learning for Biology](../../Tier_3_Applied_Bioinformatics/10_Deep_Learning_for_Biology/01_deep_learning_for_biology.ipynb) | [Next: Tier 4 Skills Check: Algorithms & Data Structures →](../../Tier_4_Algorithms_and_Data_Structures/00_Skills_Check/00_skills_check.ipynb)